In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_from_disk

In [2]:
dataset = load_from_disk("../data/sample_dataset")

dataset

Parameter 'format_kwargs'={} of the transform datasets.arrow_dataset.Dataset.set_format couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


DatasetDict({
    train: Dataset({
        features: ['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id'],
        num_rows: 16153
    })
    validation: Dataset({
        features: ['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id'],
        num_rows: 9094
    })
})

In [3]:
train_df = dataset["train"].to_pandas()
test_df = dataset["validation"].to_pandas()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Total patents:", len(train_df) + len(test_df))

Train shape: (16153, 14)
Test shape: (9094, 14)
Total patents: 25247


# Extended EDA — Patent Intelligence

This notebook continues the exploratory data analysis of the HUPD sample
dataset after the initial dataset inspection.

## Objectives

1. Text characteristics across decision classes
2. Duplicate and near-duplicate analysis
3. Temporal distribution of patents
4. Retrieval-oriented analysis

The purpose of this EDA is not to build a prediction model. It is to
understand the corpus and identify characteristics that may influence
semantic retrieval, text representation, embedding, and evaluation.

In [4]:
train_df["decision"].value_counts()

decision
PENDING     7434
ACCEPTED    6945
REJECTED    1774
Name: count, dtype: int64

In [5]:
decision_distribution = (
    train_df["decision"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

decision_distribution

decision
PENDING     46.02
ACCEPTED    43.00
REJECTED    10.98
Name: proportion, dtype: float64

In [6]:
text_columns = [
    "title",
    "abstract",
    "claims",
    "background",
    "summary",
    "description"
]

In [7]:
decision_text_stats = {}

for decision in train_df["decision"].unique():
    print(f"Processing: {decision}")

    subset = train_df[train_df["decision"] == decision]

    decision_text_stats[decision] = {}

    for col in text_columns:
        words = (
            subset[col]
            .fillna("")
            .str.count(r"\S+")
        )

        decision_text_stats[decision][col] = {
            "mean": words.mean(),
            "median": words.median(),
            "95th_percentile": words.quantile(0.95)
        }

        del words

Processing: ACCEPTED
Processing: PENDING
Processing: REJECTED


In [9]:
rows = []

for decision in decision_text_stats:
    for col in text_columns:
        stats = decision_text_stats[decision][col]

        rows.append({
            "decision": decision,
            "section": col,
            "mean_words": stats["mean"],
            "median_words": stats["median"],
            "95th_percentile": stats["95th_percentile"]
        })

decision_text_stats_df = pd.DataFrame(rows)

decision_text_stats_df

,decision,section,mean_words,median_words,95th_percentile
0,ACCEPTED,title,7.735061,7.0,16.00
1,ACCEPTED,abstract,111.772642,116.0,157.00
2,ACCEPTED,claims,993.242477,887.0,2015.40
3,ACCEPTED,background,403.850972,283.0,1174.00
4,ACCEPTED,summary,780.020734,449.0,2551.20
5,ACCEPTED,description,9489.533045,7107.0,23992.20
6,PENDING,title,7.988970,7.0,16.00
7,PENDING,abstract,108.132096,110.0,161.00
8,PENDING,claims,1015.632634,895.0,2028.00
9,PENDING,background,406.752758,273.0,1238.05


In [10]:
print("Total rows:", len(train_df))
print("Unique patent numbers:", train_df["patent_number"].nunique())
print("Duplicate patent numbers:", train_df["patent_number"].duplicated().sum())

Total rows: 16153
Unique patent numbers: 16153
Duplicate patent numbers: 0


In [11]:
print("Duplicate titles:",
      train_df["title"].duplicated().sum())

Duplicate titles: 867


In [12]:
title_abstract = (
    train_df["title"].fillna("")
    + " "
    + train_df["abstract"].fillna("")
)

print(
    "Duplicate Title + Abstract:",
    title_abstract.duplicated().sum()
)

Duplicate Title + Abstract: 184


In [13]:
train_patents = set(train_df["patent_number"])
validation_patents = set(test_df["patent_number"])

overlap = train_patents.intersection(validation_patents)

print("Patent numbers shared between train and validation:",
      len(overlap))

Patent numbers shared between train and validation: 0


In [14]:
train_ta = set(
    (
        train_df["title"].fillna("")
        + " "
        + train_df["abstract"].fillna("")
    )
)

validation_ta = set(
    (
        test_df["title"].fillna("")
        + " "
        + test_df["abstract"].fillna("")
    )
)

text_overlap = train_ta.intersection(validation_ta)

print(
    "Exact Title + Abstract matches between train and validation:",
    len(text_overlap)
)

Exact Title + Abstract matches between train and validation: 15


In [15]:
duplicate_checks = {
    "Patent number": train_df["patent_number"].duplicated().sum(),
    "Title": train_df["title"].duplicated().sum(),
    "Abstract": train_df["abstract"].duplicated().sum(),
    "Claims": train_df["claims"].duplicated().sum(),
    "Title + Abstract": title_abstract.duplicated().sum()
}

for name, count in duplicate_checks.items():
    percentage = count / len(train_df) * 100
    print(f"{name:20s}: {count:6,} ({percentage:.2f}%)")

Patent number       :      0 (0.00%)
Title               :    867 (5.37%)
Abstract            :    234 (1.45%)
Claims              :     16 (0.10%)
Title + Abstract    :    184 (1.14%)


In [16]:
overlap_df = train_df[
    title_abstract.isin(text_overlap)
][
    ["patent_number", "title", "abstract", "decision"]
].copy()

overlap_df

,patent_number,title,abstract,decision
2385,14904892,"APPARATUS FOR TRANSMITTING BROADCAST SIGNALS, ...",The present invention provides an apparatus of...,ACCEPTED
2730,14905242,COPOLYMER AND ORGANIC SOLAR CELL COMPRISING SAME,The present specification provides a copolymer...,ACCEPTED
3139,14905659,"APPARATUS FOR TRANSMITTING BROADCAST SIGNALS, ...",The present invention provides a method of tra...,ACCEPTED
3172,14905692,"APPARATUS FOR TRANSMITTING BROADCAST SIGNALS, ...",The present invention provides a method of tra...,ACCEPTED
4540,14987226,Purity Analysis Using White List/Black List An...,Memoizable functions may be identified by anal...,ACCEPTED
5590,14988447,Device and Method for Through the Scope Endosc...,Medical device used to cause hemostasis of blo...,PENDING
5908,14988827,INDEPENDENT BEACON BASED LIGHT POSITION SYSTEM,Systems an methods are provided that disclose ...,ACCEPTED
6706,14989742,MULTISUBSTITUTED AROMATIC COMPOUNDS AS INHIBIT...,There are provided inter alia multisubstituted...,REJECTED
8138,14991379,Processing of Portable Device Data,A method for processing data in a data process...,PENDING
9172,14992568,SMARTPHONE,A smartphone includes: a cover layer; a displa...,ACCEPTED


In [17]:
validation_overlap_df = test_df[
    (
        test_df["title"].fillna("")
        + " "
        + test_df["abstract"].fillna("")
    ).isin(text_overlap)
][
    ["patent_number", "title", "abstract", "decision"]
].copy()

validation_overlap_df

,patent_number,title,abstract,decision
1130,14908016,"APPARATUS FOR TRANSMITTING BROADCAST SIGNALS, ...",The present invention provides an apparatus of...,ACCEPTED
1408,14908304,"APPARATUS FOR TRANSMITTING BROADCAST SIGNALS, ...",The present invention provides a method of tra...,PENDING
1513,14908410,"APPARATUS FOR TRANSMITTING BROADCAST SIGNALS, ...",The present invention provides a method of tra...,ACCEPTED
1535,14908433,"APPARATUS FOR TRANSMITTING BROADCAST SIGNALS, ...",The present invention provides a method of tra...,ACCEPTED
1862,14908771,"APPARATUS FOR TRANSMITTING BROADCAST SIGNALS, ...",The present invention provides an apparatus of...,PENDING
1915,14908826,"APPARATUS FOR TRANSMITTING BROADCAST SIGNALS, ...",The present invention provides an apparatus of...,ACCEPTED
2062,14908979,COPOLYMER AND ORGANIC SOLAR CELL COMPRISING SAME,The present specification provides a copolymer...,ACCEPTED
2073,14908991,"APPARATUS FOR TRANSMITTING BROADCAST SIGNALS, ...",The present invention provides a method of tra...,ACCEPTED
2882,15004528,Maleinized Ester Derivatives,This disclosed invention relates to a maleinat...,ACCEPTED
3109,15004789,MULTISUBSTITUTED AROMATIC COMPOUNDS AS INHIBIT...,There are provided inter alia multisubstituted...,ACCEPTED


In [18]:
# Recreate the combined representation
train_ta_series = (
    train_df["title"].fillna("")
    + " "
    + train_df["abstract"].fillna("")
)

validation_ta_series = (
    test_df["title"].fillna("")
    + " "
    + test_df["abstract"].fillna("")
)

# Find representations shared between the two splits
shared_representations = (
    set(train_ta_series)
    .intersection(set(validation_ta_series))
)

rows = []

for representation in shared_representations:

    train_matches = train_df[
        train_ta_series == representation
    ]

    validation_matches = test_df[
        validation_ta_series == representation
    ]

    rows.append({
        "title": train_matches.iloc[0]["title"],
        "train_count": len(train_matches),
        "validation_count": len(validation_matches),
        "train_patents": list(train_matches["patent_number"]),
        "validation_patents": list(validation_matches["patent_number"])
    })

shared_df = pd.DataFrame(rows)

shared_df

,title,train_count,validation_count,train_patents,validation_patents
0,Processing of Portable Device Data,1,1,[14991379],[15008296]
1,"RESIST COMPOSITION, METHOD FOR FORMING RESIST ...",1,1,[15000403],[15005177]
2,Device and Method for Through the Scope Endosc...,1,1,[14988447],[15009358]
3,PHARMACEUTICAL COMPOSITION CONTAINING A STABIL...,1,6,[14994913],"[15005753, 15005808, 15005863, 15005911, 15005..."
4,LOW NOISE TURBINE FOR GEARED TURBOFAN ENGINE,1,1,[14996544],[15007784]
5,Purity Analysis Using White List/Black List An...,1,1,[14987226],[15005643]
6,OPTICAL SENSOR,2,2,"[14997261, 14997281]","[15006124, 15007130]"
7,COPOLYMER AND ORGANIC SOLAR CELL COMPRISING SAME,1,1,[14905242],[14908979]
8,SMARTPHONE,1,1,[14992568],[15007240]
9,"APPARATUS FOR TRANSMITTING BROADCAST SIGNALS, ...",2,4,"[14905659, 14905692]","[14908304, 14908410, 14908433, 14908991]"


In [19]:
affected_train = train_ta_series.isin(shared_representations).sum()
affected_validation = validation_ta_series.isin(shared_representations).sum()

print("Shared representations:", len(shared_representations))

print(
    "Affected train patents:",
    affected_train,
    f"({affected_train / len(train_df) * 100:.2f}%)"
)

print(
    "Affected validation patents:",
    affected_validation,
    f"({affected_validation / len(test_df) * 100:.2f}%)"
)

Shared representations: 15
Affected train patents: 17 (0.11%)
Affected validation patents: 28 (0.31%)


## Duplicate Analysis — Findings

The training split contains 16,153 patent records, with no duplicate
patent numbers. However, exact textual duplication exists within the
dataset.

The observed duplicate rates are:

| Field | Duplicate Records | Percentage |
|---|---:|---:|
| Patent number | 0 | 0.00% |
| Title | 867 | 5.37% |
| Abstract | 234 | 1.45% |
| Claims | 16 | 0.10% |
| Title + Abstract | 184 | 1.14% |

No patent-number overlap was found between the training and validation
splits.

However, 15 unique Title + Abstract representations occur in both
splits. These representations correspond to 17 training patents
(0.11% of the training set) and 28 validation patents (0.31% of the
validation set).

Several shared representations correspond to multiple patent records.
Therefore, identical Title + Abstract text does not necessarily imply
that the same patent occurs in both splits.

### Implication for the Patent Intelligence pipeline

The observed overlap is small and does not currently justify modifying
or removing records from the dataset. However, it should be considered
when constructing retrieval evaluation datasets. In particular,
evaluation should account for cases where multiple distinct patent
records have identical textual representations.

Option A — TF-IDF candidate detection
Patent text
    ↓
TF-IDF
    ↓
sparse representation
    ↓
similarity search
    ↓
high similarity candidates

Cheap enough for your machine.

Option B — Embedding-based detection
Patent text
    ↓
Embedding model
    ↓
vector representation
    ↓
FAISS / similarity search
    ↓
near-duplicate candidates

This is actually closer to our eventual pipeline, but I'd save it for the embedding stage.

For EDA, I'd use TF-IDF only as a lightweight near-duplicate investigation, and avoid generating embeddings just for EDA.

So the next section can be:

## 2. Duplicate / Near-Duplicate Analysis

### 2.1 Exact duplicates       ✅
### 2.2 Cross-split overlap    ✅
### 2.3 Near-duplicates        ← next

And we'll do the near-duplicate search on Title + Abstract first, rather than the entire Description, because your Description can reach hundreds of thousands of words and is exactly the kind of thing that could blow up memory.

Next we'll build a memory-conscious near-duplicate check.